# NetworksProject Multi-Machine Runbook (2 Machines)

This notebook is for running the project over a local network on **two machines** and collecting clean evidence for your video demo.

Use the same repository and matching config files on both machines.

Expected peers in `PeerInfo2.cfg`:
- `1001` is the seeder (`has_file=1`)
- `1002` is the downloader (`has_file=0`)

## 1) Load Project Paths and Parse `PeerInfo2.cfg`

Reads config from repo root and normalizes peer rows into fields:
`peer_id`, `ip`, `port`, `has_file`.

In [ ]:
from pathlib import Path
import ipaddress
import json
import os
import socket
import subprocess
import sys
import time
import threading
from datetime import datetime

REPO_ROOT = Path.cwd()
PEERINFO_FILE = REPO_ROOT / "PeerInfo2.cfg"
COMMON_FILE = REPO_ROOT / "Common.cfg"

assert PEERINFO_FILE.exists(), f"Missing: {PEERINFO_FILE}"
assert COMMON_FILE.exists(), f"Missing: {COMMON_FILE}"


def parse_peerinfo(path: Path):
    peers = []
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line:
            continue
        pid, host, port, has_file = line.split()
        peers.append({
            "peer_id": int(pid),
            "ip": host,
            "port": int(port),
            "has_file": int(has_file),
        })
    return peers


def parse_common(path: Path):
    cfg = {}
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line:
            continue
        key, value = line.split()
        cfg[key] = int(value) if value.isdigit() else value
    return cfg


peers = parse_peerinfo(PEERINFO_FILE)
common_cfg = parse_common(COMMON_FILE)

print(f"Repo root: {REPO_ROOT}")
print(f"Peer config: {PEERINFO_FILE.name}")
print("Parsed peers:")
for p in peers:
    print(p)

## 2) Validate IP/Port Mapping for Two Machines

Checks exactly two peers, valid IPs, numeric ports, and address uniqueness.

In [ ]:
errors = []

if len(peers) != 2:
    errors.append(f"Expected exactly 2 peers, found {len(peers)}")

for p in peers:
    try:
        ipaddress.ip_address(p["ip"])
    except ValueError:
        errors.append(f"Invalid IP for peer {p['peer_id']}: {p['ip']}")

    if not isinstance(p["port"], int) or p["port"] < 1 or p["port"] > 65535:
        errors.append(f"Invalid port for peer {p['peer_id']}: {p['port']}")

pair_set = {(p["ip"], p["port"]) for p in peers}
if len(pair_set) != len(peers):
    errors.append("Duplicate (ip, port) pair detected in PeerInfo2.cfg")

if not any(p["has_file"] == 1 for p in peers):
    errors.append("At least one peer must have has_file=1")

for p in peers:
    ip_obj = ipaddress.ip_address(p["ip"])
    if not ip_obj.is_private:
        print(f"Warning: peer {p['peer_id']} uses non-private IP {p['ip']} (okay if your LAN routes it)")

if errors:
    print("Validation FAILED:")
    for e in errors:
        print(f"- {e}")
    raise SystemExit("Fix PeerInfo2.cfg before launch.")

print("Validation PASSED for two-machine setup.")

## 3) Auto-Detect Local Machine and Select Peer ID

Attempts to match this machine's IP(s) to one row in `PeerInfo2.cfg`.

If auto-detect fails because of multiple adapters, set `MANUAL_PEER_ID`.

In [ ]:
MANUAL_PEER_ID = None  # set to 1001 or 1002 if needed


def get_local_ips():
    ips = set()
    try:
        host = socket.gethostname()
        for rec in socket.getaddrinfo(host, None, family=socket.AF_INET):
            ips.add(rec[4][0])
    except Exception:
        pass

    # UDP trick: gets the outbound local IP without sending payload
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        ips.add(s.getsockname()[0])
        s.close()
    except Exception:
        pass

    ips.add("127.0.0.1")
    return sorted(ips)


local_ips = get_local_ips()
print("Local IPv4 candidates:", local_ips)

matched = [p for p in peers if p["ip"] in local_ips]
if MANUAL_PEER_ID is not None:
    me = next((p for p in peers if p["peer_id"] == MANUAL_PEER_ID), None)
    assert me is not None, f"MANUAL_PEER_ID {MANUAL_PEER_ID} not in PeerInfo2.cfg"
elif len(matched) == 1:
    me = matched[0]
else:
    print("Auto-detect could not uniquely match this machine.")
    print("Set MANUAL_PEER_ID = 1001 or 1002 and re-run this cell.")
    raise SystemExit("Peer auto-selection failed")

other = next(p for p in peers if p["peer_id"] != me["peer_id"])
print(f"This machine selected peer_id={me['peer_id']} ({me['ip']}:{me['port']})")
print(f"Remote peer is peer_id={other['peer_id']} ({other['ip']}:{other['port']})")

## 4) Generate Machine-Specific Runtime Config

Creates a per-machine runtime folder with launch metadata and log paths.

In [ ]:
runtime_dir = REPO_ROOT / "runtime" / f"peer_{me['peer_id']}"
runtime_dir.mkdir(parents=True, exist_ok=True)

runtime = {
    "selected_peer": me,
    "other_peer": other,
    "peerinfo_file": str(PEERINFO_FILE),
    "common_file": str(COMMON_FILE),
    "stdout_log": str(runtime_dir / "stdout.log"),
    "merged_video_log": str(runtime_dir / "video_evidence.log"),
    "created_at": datetime.now().isoformat(timespec="seconds"),
}

runtime_json = runtime_dir / "runtime.json"
runtime_json.write_text(json.dumps(runtime, indent=2), encoding="utf-8")

print(f"Runtime folder: {runtime_dir}")
print(f"Runtime metadata: {runtime_json}")
print(json.dumps(runtime, indent=2))

## 5) Run Connectivity Checks (Ping + TCP Port Probe)

Verifies LAN reachability to the other machine before launch.

In [ ]:
def ping_host(ip: str, count: int = 2):
    cmd = ["ping", "-n", str(count), ip]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    return proc.returncode == 0, (proc.stdout + "\n" + proc.stderr).strip()


def tcp_probe(ip: str, port: int, timeout_s: float = 2.0):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(timeout_s)
    try:
        s.connect((ip, port))
        s.close()
        return True, "TCP connect succeeded"
    except Exception as exc:
        return False, f"TCP connect failed: {exc}"


ok_ping, ping_out = ping_host(other["ip"])
ok_tcp, tcp_out = tcp_probe(other["ip"], other["port"])

print(f"Ping to {other['ip']}: {'PASS' if ok_ping else 'FAIL'}")
print(f"TCP probe to {other['ip']}:{other['port']}: {'PASS' if ok_tcp else 'FAIL'}")
print("\nTCP details:", tcp_out)

if not ok_ping:
    print("Hint: check both machines are on same network and IPs are correct in PeerInfo2.cfg")
if not ok_tcp:
    print("Hint: start the other peer first (ID order), open firewall for the peer port, and verify target IP/port")

## 6) Launch Peer Process on Machine A

Run this on the machine whose selected `peer_id` is `1001`.

In [ ]:
PROC = None


def launch_selected_peer():
    global PROC
    if PROC is not None and PROC.poll() is None:
        print("Peer process already running.")
        return PROC

    cmd = [sys.executable, "peerProcess.py", str(me["peer_id"]), "PeerInfo2.cfg"]
    out_path = Path(runtime["stdout_log"])
    logf = open(out_path, "a", encoding="utf-8")
    PROC = subprocess.Popen(
        cmd,
        cwd=str(REPO_ROOT),
        stdout=logf,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print("Launched:", " ".join(cmd))
    print("PID:", PROC.pid)
    print("stdout log:", out_path)
    return PROC


if me["peer_id"] == 1001:
    launch_selected_peer()
else:
    print("This machine is not peer 1001. Skip this cell and use section 7.")

## 7) Launch Peer Process on Machine B

Run this on the machine whose selected `peer_id` is `1002`.

A small delay is included so Machine A can bind first.

In [ ]:
STARTUP_DELAY_SECONDS = 2
print(f"Command template: {sys.executable} peerProcess.py <peer_id> PeerInfo2.cfg")

if me["peer_id"] == 1002:
    print(f"Waiting {STARTUP_DELAY_SECONDS}s before launching peer 1002...")
    time.sleep(STARTUP_DELAY_SECONDS)
    launch_selected_peer()
else:
    print("This machine is not peer 1002. Section 6 is your launch cell.")

## 8) Stream and Save Logs for Video Evidence

Tails current run logs and writes timestamped lines to a consolidated evidence file.

In [ ]:
def tail_file(src: Path, dst: Path, seconds: int = 20):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)

    if not src.exists():
        print(f"No source log yet: {src}")
        return

    start = time.time()
    with src.open("r", encoding="utf-8", errors="replace") as fin, dst.open("a", encoding="utf-8") as fout:
        fin.seek(0, os.SEEK_END)
        print(f"Streaming {src.name} for {seconds}s -> {dst.name}")
        while time.time() - start < seconds:
            line = fin.readline()
            if line:
                stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                merged = f"[{stamp}] {line.rstrip()}"
                print(merged)
                fout.write(merged + "\n")
                fout.flush()
            else:
                time.sleep(0.2)


tail_file(Path(runtime["stdout_log"]), Path(runtime["merged_video_log"]), seconds=25)
print("Saved evidence log:", runtime["merged_video_log"])

## 9) Verify Handshake and Piece Exchange Events

Scans log files for expected milestones and prints a pass/fail checklist.

In [ ]:
def read_text_if_exists(path: Path):
    return path.read_text(encoding="utf-8", errors="replace") if path.exists() else ""

local_log = read_text_if_exists(Path(runtime["stdout_log"]))
peer_log_file = REPO_ROOT / f"log_peer_{me['peer_id']}.log"
protocol_log = read_text_if_exists(peer_log_file)
combined = (local_log + "\n" + protocol_log).lower()

checks = {
    "tcp_connected": ["makes a connection", "is connected from"],
    "unchoked": ["is unchoked by"],
    "piece_download": ["has downloaded the piece"],
    "complete_file": ["has downloaded the complete file"],
}

print("Checklist:")
for name, needles in checks.items():
    ok = any(n in combined for n in needles)
    print(f"- {name}: {'PASS' if ok else 'MISSING'}")

print("\nProtocol log searched:", peer_log_file)

## 10) Quick Retry Workflow for Common Local-Network Failures

Stops local peer process, kills stale `peerProcess.py`, reruns preflight checks, and prints relaunch command.

In [ ]:
def stop_local_proc():
    global PROC
    if PROC is not None and PROC.poll() is None:
        PROC.terminate()
        try:
            PROC.wait(timeout=5)
        except Exception:
            PROC.kill()
        print(f"Stopped local process PID {PROC.pid}")
    else:
        print("No local notebook-managed process running.")


def kill_stale_peerprocess_windows():
    cmd = [
        "powershell",
        "-NoProfile",
        "-Command",
        "Get-CimInstance Win32_Process | Where-Object { $_.Name -eq 'python.exe' -and $_.CommandLine -match 'peerProcess.py' } | ForEach-Object { Stop-Process -Id $_.ProcessId -Force }",
    ]
    subprocess.run(cmd, capture_output=True, text=True)
    print("Requested stop for stale peerProcess.py python processes.")


stop_local_proc()
kill_stale_peerprocess_windows()

ok_ping, _ = ping_host(other["ip"])
ok_tcp, tcp_info = tcp_probe(other["ip"], other["port"])
print(f"Re-check ping: {'PASS' if ok_ping else 'FAIL'}")
print(f"Re-check tcp: {'PASS' if ok_tcp else 'FAIL'} ({tcp_info})")
print("Relaunch command:")
print(f"{sys.executable} peerProcess.py {me['peer_id']} PeerInfo2.cfg")

## Demo Recording Checklist

1. Confirm both machines use the same `Common.cfg` and `PeerInfo2.cfg`.
2. Ensure machine for peer `1001` has file at `peer_1001/thefile`.
3. Open firewall for configured peer port on both machines.
4. Start machine `1001` first, then machine `1002`.
5. Capture section 8 output and `log_peer_*.log` as evidence.
6. End by showing section 9 checklist and completed file on peer `1002`.